# Ejercicio: Chatbot E-commerce Alimentación - Gourmet Express

Este notebook contiene la implementación de un sistema **RAG (Retrieval-Augmented Generation)** para un chatbot de una tienda gourmet. El bot es capaz de responder sobre:
- Catálogo de productos (precios y detalles).
- Políticas de devolución y envío.
- Promociones y cupones de descuento.

In [1]:
import os
from getpass import getpass
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Configuración de API Key (si no está ya en el entorno)
if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass("Introduce tu GROQ API Key: ")

print("Configuración lista ✓")

Configuración lista ✓


### 1. Definición de la Base de Conocimiento
Aquí definimos los datos de nuestra tienda: productos, políticas y promociones.

In [2]:
ecommerce_docs = [
    # Productos
    Document(
        page_content="Aceite de Oliva Virgen Extra (AOVE) Premium - Botella 500ml. Origen: Jaén, España. Precio: 12.50€. Nota de cata: Frutado intenso con toques de hierba recién cortada.", 
        metadata={"categoria": "aceites", "id": "p001"}
    ),
    Document(
        page_content="Jamón Ibérico de Bellota 100% Raza Ibérica - Sobre de 100g cortado a mano. Precio: 18.90€. Curación artesanal de más de 36 meses en Guijuelo.", 
        metadata={"categoria": "charcuteria", "id": "p002"}
    ),
    Document(
        page_content="Queso Manchego Curado DOP - Cuña de 250g. Elaborado con leche cruda de oveja manchega. Precio: 8.75€. Sabor potente y persistente.", 
        metadata={"categoria": "quesos", "id": "p003"}
    ),
    Document(
        page_content="Vino Tinto Reserva Rioja - Botella 750ml. Uva Tempranillo. Precio: 15.00€. 18 meses en barrica de roble americano.", 
        metadata={"categoria": "vinos", "id": "p004"}
    ),
    
    # Políticas
    Document(
        page_content="Política de devoluciones: Al tratarse de productos alimenticios perecederos, solo se aceptan devoluciones en caso de producto defectuoso, en mal estado o error en el envío. Se debe notificar en un plazo máximo de 24 horas desde la recepción del pedido.", 
        metadata={"tipo": "politica", "tema": "devoluciones"}
    ),
    Document(
        page_content="Condiciones de Envío: Los pedidos se entregan en 24-48h laborables. El envío es gratuito en pedidos superiores a 60€ para toda la península.", 
        metadata={"tipo": "politica", "tema": "envio"}
    ),
    
    # Promociones
    Document(
        page_content="Cupón BIENVENIDA: Los nuevos clientes pueden usar el código 'GOURMET10' para obtener un 10% de descuento en su primera compra.", 
        metadata={"tipo": "promocion", "codigo": "GOURMET10"}
    ),
    Document(
        page_content="Oferta Temporal: 3x2 en todos los quesos manchegos durante esta semana. La unidad más barata sale gratis.", 
        metadata={"tipo": "promocion", "tema": "3x2"}
    ),
]

print(f"Base de conocimiento creada con {len(ecommerce_docs)} documentos.")

Base de conocimiento creada con 8 documentos.


### 2. Creación del Sistema RAG

In [3]:
# Cargar modelo de embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Crear Vector Store
vector_store = FAISS.from_documents(ecommerce_docs, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Configurar LLM
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.3)

print("Sistema RAG inicializado ✓")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sistema RAG inicializado ✓


### 3. Prompt y Lógica del Chatbot

In [4]:
ecommerce_system_prompt = """Eres 'GourmetBot', el sumiller y asistente experto de la tienda Gourmet Express. 
Tu objetivo es ayudar a los clientes con información sobre productos, guiarles en el proceso de compra y resolver dudas sobre envíos o promociones.

Instrucciones:
1. Usa el contexto proporcionado para responder de forma elegante y profesional.
2. Si un cliente pregunta por un producto que no está en el contexto, sugiere que contacte con nuestro equipo humano en soporte@gourmetexpress.es.
3. Si hay una promoción aplicable (como el cupón de bienvenida o el 3x2), menciónala si es relevante para la pregunta.

Contexto:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", ecommerce_system_prompt),
    ("human", "{question}")
])

def gourmet_chatbot(pregunta: str):
    # 1. Recuperar información relevante
    docs = retriever.invoke(pregunta)
    contexto = "\n\n".join([d.page_content for d in docs])
    
    # 2. Generar respuesta con LLM
    mensajes = prompt.format_messages(context=contexto, question=pregunta)
    respuesta = llm.invoke(mensajes)
    
    print(f"👤 Cliente: {pregunta}")
    print(f"🤖 GourmetBot: {respuesta.content}")
    print("-" * 50)

### 4. Pruebas

In [5]:
gourmet_chatbot("¿Qué jamón tenéis y cuánto cuesta?")
gourmet_chatbot("¿Tenéis alguna oferta en quesos?")
gourmet_chatbot("He recibido un producto en mal estado, ¿puedo devolverlo?")
gourmet_chatbot("¿Tenéis caviar?")

👤 Cliente: ¿Qué jamón tenéis y cuánto cuesta?
🤖 GourmetBot: En Gourmet Express, tenemos un delicioso Jamón Ibérico de Bellota 100% Raza Ibérica, cortado a mano en sobres de 100g. Este producto de alta calidad ha sido curado artesanalmente durante más de 36 meses en Guijuelo, lo que le da un sabor y textura únicos.

El precio de este jamón es de 18,90€ por sobre de 100g. Si estás buscando un producto de alta calidad y auténtico sabor ibérico, este jamón es una excelente opción. ¿Te gustaría saber más sobre este producto o necesitas ayuda con algo más?
--------------------------------------------------
👤 Cliente: ¿Tenéis alguna oferta en quesos?
🤖 GourmetBot: ¡Claro que sí! En Gourmet Express, estamos ofreciendo una promoción especial en nuestros quesos manchegos durante esta semana. Puedes disfrutar de un 3x2 en todos nuestros quesos manchegos, lo que significa que la unidad más barata sale gratis. ¡Es una excelente oportunidad para probar diferentes variedades o surtir tu despensa con 